# Combine and curate the melting-point sources

This notebook builds the combined melting-point dataset from the four sources in `input_datasets/`, up to (but not including) the stereo clean-up, which is applied in `2_stereo_cleanup.ipynb`.

Sources, in their established order: 1 = ChemXplore (Marimuthu, 2025), 2 = EGNN (Huang, 2025), 3 = PATENTS/OCHEM (Tetko, 2016), and 4 = Bradley data from Austermeier (2025).

1. **Prepare Source 4.** Source 4 is assembled from the original study files in `input_datasets/4_Austermeier_2025/` and saved as `outputs/4_Austermeier_2025_processed.csv`. When that file already exists it is reused; set `REBUILD_SOURCE4 = True` to rebuild it from the study files.
2. **Combine and curate** all four sources: MP parsing, canonical isomeric SMILES, broad-organic structure screen, MP consensus, and the final MP and MW filters.

Outputs in `outputs/`:

- `4_Austermeier_2025_processed.csv`: the prepared Source 4 observations (`SMILES`, `MP`);
- `combined_data_before_stereo_cleanup.parquet`: resolved, broad-organic, single-component compounds with consensus `MP` in 0–500 °C and RDKit average molecular weight `MW < 1000` Da;
- `multiple_MP_compounds.csv`: an observation-level audit of canonical compounds that have more than one distinct reported MP before the final MP-range and MW filters.

In [1]:
from pathlib import Path
import hashlib
import re
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors

RDLogger.DisableLog('rdApp.*')

data_dir_candidates = [Path.cwd(), Path.cwd() / '0_data', Path.cwd().parent / '0_data']
DATA_DIR = next((path.resolve() for path in data_dir_candidates
                 if (path / 'input_datasets').is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        'Could not locate 0_data/input_datasets. Run this notebook from the repository root '
        'or from the 0_data directory.'
    )

INPUT_DIR = DATA_DIR / 'input_datasets'
OUTPUT_DIR = DATA_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Source 4: original study files and the prepared table built from them.
REBUILD_SOURCE4 = False  # True rebuilds the prepared table even if it already exists.
SOURCE4_RAW_DIR = INPUT_DIR / '4_Austermeier_2025'
SOURCE4_TRAIN_DIR = SOURCE4_RAW_DIR / 'train_without_data_augmentation'
SOURCE4_TEST_FILE = SOURCE4_RAW_DIR / 'test_predictions' / 'consensus_without_data_augmentation.csv'
SOURCE4_VALIDATION_DIR = SOURCE4_RAW_DIR / 'val'
SOURCE4_PREPARED = OUTPUT_DIR / '4_Austermeier_2025_processed.csv'

OUTPUT_DATA_BEFORE_STEREO_CLEANUP = OUTPUT_DIR / 'combined_data_before_stereo_cleanup.parquet'
OUTPUT_CONFLICTS = OUTPUT_DIR / 'multiple_MP_compounds.csv'
MAX_MW = 1000.0

INPUT_SPECS = [
    (1, INPUT_DIR / '1_Marimuthu_2025_tmpC.csv', 'tmp/ºC', 'chemxplore'),
    (2, INPUT_DIR / '2_Huang_2025_mp.csv', 'Melt_C', 'numeric'),
    (3, INPUT_DIR / '3_Tetko_2016_ochem.parquet',
     'Melting Point {measured, converted}', 'numeric_or_range'),
    (4, SOURCE4_PREPARED, 'MP', 'numeric'),
]

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

print(f'Input directory: {INPUT_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Source 4 table: {SOURCE4_PREPARED.relative_to(DATA_DIR)} '
      f'({"rebuild" if REBUILD_SOURCE4 or not SOURCE4_PREPARED.is_file() else "reuse existing file"})')

Input directory: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/0_data/input_datasets
Output directory: /Users/sdl5_mp/Documents/GitHub/melting_point_2026/0_data/outputs
Source 4 table: outputs/4_Austermeier_2025_processed.csv (rebuild)


## Prepare Source 4: Bradley data from Austermeier et al. (2025)

Source 4 is assembled from the original study files in `input_datasets/4_Austermeier_2025/`:

- all ten files in `train_without_data_augmentation/`;
- `test_predictions/consensus_without_data_augmentation.csv`, from which only `SMILES` and the experimental `exp MP` field (renamed to `MP`) are used.

Only experimental melting-point observations are retained; prediction columns and the data-augmented files are not used. The inherited train/test labels describe the source study and are not used as this project's model split. The ten validation folds are not appended: they are cross-validation partitions of the same training pool, and a check confirms that they add no SMILES absent from the training-fold union.

Deduplication is performed on the exact pair `SMILES` + `MP`. Repeated copies of the same observation are removed, while different reported MPs for the same SMILES are kept for the consensus step below. SMILES are not canonicalized here.

`build_source4()` holds the complete preparation. It runs only when `outputs/4_Austermeier_2025_processed.csv` is missing or `REBUILD_SOURCE4 = True`; otherwise the saved table is reused. Either way, the table is validated before it is used.

In [2]:
def training_fold_number(path):
    match = re.fullmatch(r'train(\d+)_curated\.csv', path.name)
    if match is None:
        raise ValueError(f'Unexpected training filename: {path.name}')
    return int(match.group(1))


def build_source4():
    # Non-augmented training folds, in fold order: train1_curated.csv ... train10_curated.csv.
    train_files = sorted(SOURCE4_TRAIN_DIR.glob('train*_curated.csv'), key=training_fold_number)
    if not train_files:
        raise FileNotFoundError(f'No training CSV files found in {SOURCE4_TRAIN_DIR}')
    fold_numbers = [training_fold_number(path) for path in train_files]
    if fold_numbers != list(range(1, 11)):
        raise ValueError(f'Expected training folds 1–10, found: {fold_numbers}')

    training_frames = []
    training_audit_rows = []
    for path in train_files:
        frame = pd.read_csv(path, usecols=['SMILES', 'MP'])
        frame['MP'] = pd.to_numeric(frame['MP'], errors='raise')
        frame['Original_subset'] = 'train'
        frame['Original_file'] = path.name
        training_frames.append(frame)
        training_audit_rows.append({
            'Fold': training_fold_number(path),
            'Original_file': path.name,
            'Rows': len(frame),
            'Unique_raw_SMILES': frame['SMILES'].nunique(dropna=False),
        })
    training_observations = pd.concat(training_frames, ignore_index=True)
    display(pd.DataFrame(training_audit_rows))
    print(f'Training rows across folds: {len(training_observations):,}')
    print(f'Unique raw training SMILES: {training_observations["SMILES"].nunique(dropna=False):,}')

    # Study test observations: only SMILES and the experimental MP are used.
    if not SOURCE4_TEST_FILE.exists():
        raise FileNotFoundError(f'Missing requested test file: {SOURCE4_TEST_FILE}')
    test_observations = pd.read_csv(SOURCE4_TEST_FILE, usecols=['SMILES', 'exp MP'])
    test_observations = test_observations.rename(columns={'exp MP': 'MP'})
    test_observations['MP'] = pd.to_numeric(test_observations['MP'], errors='raise')
    test_observations['Original_subset'] = 'test'
    test_observations['Original_file'] = SOURCE4_TEST_FILE.name
    print(f'Test rows: {len(test_observations):,}')
    print(f'Unique raw test SMILES: {test_observations["SMILES"].nunique(dropna=False):,}')

    # Combine and remove only exact duplicate SMILES–MP observations.
    all_observations = pd.concat([training_observations, test_observations], ignore_index=True)
    missing_smiles = int(all_observations['SMILES'].isna().sum())
    missing_mp = int(all_observations['MP'].isna().sum())
    if missing_smiles or missing_mp:
        raise ValueError(f'Missing required values: SMILES={missing_smiles:,}, MP={missing_mp:,}')
    exact_duplicate_mask = all_observations.duplicated(subset=['SMILES', 'MP'], keep='first')
    bradley_data = (
        all_observations.loc[~exact_duplicate_mask, ['SMILES', 'MP']]
        .sort_values(['SMILES', 'MP'], kind='stable')
        .reset_index(drop=True)
    )

    mp_counts = bradley_data.groupby('SMILES')['MP'].nunique()
    conflicting_smiles = mp_counts.loc[mp_counts > 1].index
    conflict_summary = (
        bradley_data.loc[bradley_data['SMILES'].isin(conflicting_smiles)]
        .groupby('SMILES', as_index=False)
        .agg(
            N_distinct_MP=('MP', 'nunique'),
            MP_values=('MP', lambda values: sorted(set(map(float, values)))),
            MP_min=('MP', 'min'),
            MP_max=('MP', 'max'),
        )
        .sort_values(['N_distinct_MP', 'SMILES'], ascending=[False, True], kind='stable')
        .reset_index(drop=True)
    )
    train_smiles = set(training_observations['SMILES'])
    test_smiles = set(test_observations['SMILES'])
    overlap_pairs = (
        training_observations[['SMILES', 'MP']].drop_duplicates()
        .merge(test_observations[['SMILES', 'MP']].drop_duplicates(), on=['SMILES', 'MP'], how='inner')
    )
    display(pd.DataFrame({
        'Metric': [
            'Input training rows', 'Input test rows', 'Combined input rows',
            'Exact duplicate SMILES–MP rows removed', 'Output SMILES–MP observations',
            'Unique raw SMILES in output', 'SMILES with multiple distinct MPs retained',
            'Raw SMILES shared by train and test', 'Exact SMILES–MP pairs shared by train and test',
        ],
        'Count': [
            len(training_observations), len(test_observations), len(all_observations),
            int(exact_duplicate_mask.sum()), len(bradley_data), bradley_data['SMILES'].nunique(),
            len(conflict_summary), len(train_smiles & test_smiles), len(overlap_pairs),
        ],
    }))
    display(conflict_summary)

    # The validation folds must not add structures absent from the training-fold union.
    validation_files = sorted(
        SOURCE4_VALIDATION_DIR.glob('val*_curated.csv'),
        key=lambda path: int(re.fullmatch(r'val(\d+)_curated\.csv', path.name).group(1)),
    )
    if validation_files:
        validation_smiles = set()
        for path in validation_files:
            validation_smiles.update(pd.read_csv(path, usecols=['SMILES'])['SMILES'].dropna())
        validation_only_smiles = validation_smiles - train_smiles
        print(f'Unique validation SMILES: {len(validation_smiles):,}')
        print(f'Validation SMILES absent from training-fold union: {len(validation_only_smiles):,}')
        if validation_only_smiles:
            raise ValueError(
                'Validation files contain structures absent from the training-fold union. '
                'Review the intended Source 4 coverage before writing the output.'
            )
    else:
        print('No validation files found; validation-coverage check skipped.')

    input_inventory = pd.DataFrame([
        {'Role': 'train fold', 'Original_path': str(path.relative_to(DATA_DIR)),
         'Rows': len(frame), 'SHA256': sha256(path)}
        for path, frame in zip(train_files, training_frames)
    ] + [{
        'Role': 'test observations', 'Original_path': str(SOURCE4_TEST_FILE.relative_to(DATA_DIR)),
        'Rows': len(test_observations), 'SHA256': sha256(SOURCE4_TEST_FILE),
    }])
    return bradley_data, len(conflict_summary), input_inventory

In [3]:
if REBUILD_SOURCE4 or not SOURCE4_PREPARED.is_file():
    bradley_data, n_conflicting_smiles, source4_inventory = build_source4()
    bradley_data.to_csv(SOURCE4_PREPARED, index=False)
    display(source4_inventory)
    print(f'Wrote: {SOURCE4_PREPARED.relative_to(DATA_DIR.parent)}')
else:
    bradley_data, n_conflicting_smiles = None, None
    print(f'Reusing {SOURCE4_PREPARED.relative_to(DATA_DIR.parent)}; set REBUILD_SOURCE4 = True to rebuild it.')

saved = pd.read_csv(SOURCE4_PREPARED)
saved['MP'] = pd.to_numeric(saved['MP'], errors='raise')
assert list(saved.columns) == ['SMILES', 'MP']
assert saved['SMILES'].notna().all()
assert saved['MP'].notna().all()
assert not saved.duplicated(['SMILES', 'MP']).any()
saved_mp_counts = saved.groupby('SMILES')['MP'].nunique()
if bradley_data is not None:
    assert saved.equals(bradley_data)
    assert int(saved_mp_counts.gt(1).sum()) == n_conflicting_smiles

print(f'Source 4 observations: {len(saved):,}')
print(f'Unique raw SMILES: {saved["SMILES"].nunique():,}')
print(f'SMILES retaining multiple MP values: {saved_mp_counts.gt(1).sum():,}')
print(f'SHA256: {sha256(SOURCE4_PREPARED)}')
print('All Source 4 checks passed.')

,Fold,Original_file,Rows,Unique_raw_SMILES
0,1,train1_curated.csv,13734,13715
1,2,train2_curated.csv,13734,13721
2,3,train3_curated.csv,13734,13716
3,4,train4_curated.csv,13734,13717
4,5,train5_curated.csv,13734,13717
5,6,train6_curated.csv,13734,13715
6,7,train7_curated.csv,13734,13716
7,8,train8_curated.csv,13734,13719
8,9,train9_curated.csv,13734,13720
9,10,train10_curated.csv,13734,13717


Training rows across folds: 137,340
Unique raw training SMILES: 17,633
Test rows: 1,961
Unique raw test SMILES: 1,961


,Metric,Count
0,Input training rows,137340
1,Input test rows,1961
2,Combined input rows,139301
3,Exact duplicate SMILES–MP rows removed,119692
4,Output SMILES–MP observations,19609
5,Unique raw SMILES in output,19588
6,SMILES with multiple distinct MPs retained,21
7,Raw SMILES shared by train and test,6
8,Exact SMILES–MP pairs shared by train and test,2


,SMILES,N_distinct_MP,MP_values,MP_min,MP_max
0,CC(=O)OC[C@H]1O[C@H](OC(=O)C)[C@@H]([C@H]([C@@...,2,"[111.0, 113.3]",111.0,113.3
1,CC(CCC[C@H]([C@H]1CC[C@@H]2[C@]1(C)CC[C@H]1[C@...,2,"[148.5, 149.0]",148.5,149.0
2,CO[C@H]1O[C@H](CO)[C@H]([C@@H]([C@H]1O)O)O,2,"[168.0, 169.0]",168.0,169.0
3,Nc1ccc(cc1)N=Nc1ccccc1,2,"[124.0, 127.0]",124.0,127.0
4,O=C(Nc1c(C)cccc1C)CN1CCN(CC1)CCCC(c1ccc(cc1)F)...,2,"[159.0, 160.0]",159.0,160.0
5,O=C1CC[C@H]2C(=C1)CC[C@@H]1[C@@H]2CC[C@]2([C@H...,2,"[112.0, 118.0]",112.0,118.0
6,O=C1CC[C@]2(C(=C1)CC[C@@H]1[C@@H]2CC[C@]2([C@H...,2,"[153.0, 155.0]",153.0,155.0
7,O=C1CC[C@]2(C(=C1)[C@@H](C)C[C@@H]1[C@@H]2CC[C...,2,"[206.5, 208.0]",206.5,208.0
8,OCC(=O)[C@@]1(O)CC[C@@H]2[C@]1(C)C[C@H](O)[C@H...,2,"[214.0, 223.0]",214.0,223.0
9,OCC(=O)[C@@]1(O)[C@@H](C)C[C@@H]2[C@]1(C)C[C@H...,2,"[232.0, 232.5]",232.0,232.5


Unique validation SMILES: 16,193
Validation SMILES absent from training-fold union: 0


,Role,Original_path,Rows,SHA256
0,train fold,input_datasets/4_Austermeier_2025/train_withou...,13734,4adc1acc2f5137810e583154bd530697a811ee6f20efd5...
1,train fold,input_datasets/4_Austermeier_2025/train_withou...,13734,cf67296b1c227aeffcd9ff15ee22741754a5a2495b6b91...
2,train fold,input_datasets/4_Austermeier_2025/train_withou...,13734,aa3f89409e8b084c7b42635156e6de5858bea32755b9a1...
3,train fold,input_datasets/4_Austermeier_2025/train_withou...,13734,2a5f0fafe8d40af51e5e463d9a591737942523547579f0...
4,train fold,input_datasets/4_Austermeier_2025/train_withou...,13734,681ac8d9634e5099ea9c6d814399b371051abe938f6f00...
5,train fold,input_datasets/4_Austermeier_2025/train_withou...,13734,b53de2b7f08d6eeed150214dfcdd4a132dc78d726dd89e...
6,train fold,input_datasets/4_Austermeier_2025/train_withou...,13734,b42ed52d157b58d832a514af8aa85b606f0470d0c1d871...
7,train fold,input_datasets/4_Austermeier_2025/train_withou...,13734,476f5f6f079132ba59efd65567c0afe14274817b1468b2...
8,train fold,input_datasets/4_Austermeier_2025/train_withou...,13734,50ae5c48a705ee73b5e41495cf4049b93003bc4f9b3116...
9,train fold,input_datasets/4_Austermeier_2025/train_withou...,13734,daa3faee4774d28e4256135b1ac1287dc9a4d99769eb9d...


Wrote: 0_data/outputs/4_Austermeier_2025_processed.csv
Source 4 observations: 19,609
Unique raw SMILES: 19,588
SMILES retaining multiple MP values: 21
SHA256: ecf06847acdb43c3aaec73b6d743d34862f180fd63f959484eebe77b9a6ac78e
All Source 4 checks passed.


## Combine and curate the four sources

- Canonical isomeric SMILES are generated with RDKit and must parse successfully after canonicalization.
- ChemXplore values are parsed from the original `tmp/ºC` field. Numeric values with parenthetical uncertainty are retained; censored or approximate values (`<`, `>`, `≈`) are excluded.
- Strict numeric PATENTS ranges are converted to their midpoint. The later consensus statistics describe accepted processed observations across sources, not the endpoints of an individual PATENTS range.
- Broad-organic filtering removes metal-containing and multi-component structures but retains halogen-containing compounds.
- RDKit average molecular weight is calculated from the canonical structure. The final dataset applies the strict condition `MW < 1000` Da.
- MP quality is `single`, `exact`, `near_exact` (spread ≤1 °C), `high_confidence` (>1–5 °C), or `review` (>5–10 °C).
- Groups spanning more than 10 °C are accepted only when at least two sources form a dominant cluster spanning no more than 5 °C and supporting at least two-thirds of the observations. Otherwise they remain unresolved.
- The median of accepted observations is the final `MP`. Calculations use unrounded values.


In [4]:
NUMBER = r'[+-]?(?:\d+(?:\.\d*)?|\.\d+)'
STRICT_RANGE = re.compile(rf'^\s*({NUMBER})\s*[-–]\s*({NUMBER})\s*$')
CHEMXPLORE_NUMBER = re.compile(rf'^\s*({NUMBER})(?:\(\d+\))?\s*$')

HALOGEN_ATOMIC_NUMBERS = {9, 17, 35, 53, 85, 117}
METAL_SYMBOLS = {
    'Li', 'Be', 'Na', 'Mg', 'Al', 'K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe',
    'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru',
    'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn', 'Cs', 'Ba', 'La', 'Ce', 'Pr', 'Nd', 'Pm',
    'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu', 'Hf', 'Ta', 'W',
    'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Po', 'Fr', 'Ra', 'Ac',
    'Th', 'Pa', 'U', 'Np', 'Pu', 'Am', 'Cm', 'Bk', 'Cf', 'Es', 'Fm', 'Md', 'No',
    'Lr', 'Rf', 'Db', 'Sg', 'Bh', 'Hs', 'Mt', 'Ds', 'Rg', 'Cn', 'Nh', 'Fl', 'Mc', 'Lv'
}
periodic_table = Chem.GetPeriodicTable()
METAL_ATOMIC_NUMBERS = {periodic_table.GetAtomicNumber(symbol) for symbol in METAL_SYMBOLS}

def parse_mp(value, method):
    if pd.isna(value):
        return np.nan, 'missing'
    text = str(value).strip()
    if method == 'chemxplore':
        match = CHEMXPLORE_NUMBER.fullmatch(text)
        return (float(match.group(1)), 'numeric') if match else (np.nan, 'censored_or_approximate')
    try:
        return float(text), 'numeric'
    except ValueError:
        if method == 'numeric_or_range':
            match = STRICT_RANGE.fullmatch(text)
            if match:
                return (float(match.group(1)) + float(match.group(2))) / 2.0, 'range_midpoint'
        return np.nan, 'nonnumeric'

def canonicalize_and_screen(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None, False, False, 0, np.nan
        canonical = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
        roundtrip = Chem.MolFromSmiles(canonical)
        if roundtrip is None:
            return None, False, False, 0, np.nan
        canonical = Chem.MolToSmiles(roundtrip, canonical=True, isomericSmiles=True)
        atomic_numbers = {atom.GetAtomicNum() for atom in roundtrip.GetAtoms()}
        return (
            canonical,
            bool(atomic_numbers & HALOGEN_ATOMIC_NUMBERS),
            bool(atomic_numbers & METAL_ATOMIC_NUMBERS),
            len(Chem.GetMolFrags(roundtrip)),
            float(Descriptors.MolWt(roundtrip)),
        )
    except Exception:
        return None, False, False, 0, np.nan

def quality_from_values(values, n_sources):
    spread = float(np.max(values) - np.min(values))
    if len(values) == 1 and n_sources == 1:
        return 'single'
    if spread == 0:
        return 'exact'
    if spread <= 1:
        return 'near_exact'
    if spread <= 5:
        return 'high_confidence'
    if spread <= 10:
        return 'review'
    return 'unresolved'

In [5]:
schema_rows = []
for source_id, path, mp_column, _ in INPUT_SPECS:
    if not path.is_file():
        raise FileNotFoundError(f'Missing required input file: {path}')
    columns = (pq.read_schema(path).names if path.suffix.lower() == '.parquet'
               else pd.read_csv(path, nrows=0).columns.tolist())
    missing_columns = sorted({'SMILES', mp_column}.difference(columns))
    if missing_columns:
        raise ValueError(f'{path.name} is missing required column(s): {missing_columns}')
    schema_rows.append({'Source': source_id, 'Filename': path.name,
                        'SMILES_column': 'SMILES', 'MP_column': mp_column})
display(pd.DataFrame(schema_rows))

frames = []
load_audit = []
for source_id, path, mp_column, method in INPUT_SPECS:
    filename = path.name
    if path.suffix.lower() == '.parquet':
        raw = pd.read_parquet(path, columns=['SMILES', mp_column])
    else:
        raw = pd.read_csv(path, usecols=['SMILES', mp_column], dtype={mp_column: 'string'})
    parsed = raw[mp_column].map(lambda value: parse_mp(value, method))
    frame = pd.DataFrame({
        'Original_SMILES': raw['SMILES'].astype('string'),
        'MP_raw': raw[mp_column].astype('string'),
        'MP_reported': parsed.map(lambda result: result[0]),
        'Parse_status': parsed.map(lambda result: result[1]),
        'Source': np.int8(source_id),
        'Source_file': filename,
    })
    usable = frame['Original_SMILES'].notna() & frame['MP_reported'].notna()
    load_audit.append({
        'Source_file': filename, 'Source': source_id, 'Input_rows': len(frame),
        'Usable_MP_rows': int(usable.sum()), 'Excluded_MP_rows': int((~usable).sum()),
        'Range_midpoints': int(frame['Parse_status'].eq('range_midpoint').sum()),
    })
    frames.append(frame.loc[usable])
observations = pd.concat(frames, ignore_index=True)
load_audit = pd.DataFrame(load_audit)
display(load_audit)
print(f'Usable observations before structure processing: {len(observations):,}')

,Source,Filename,SMILES_column,MP_column
0,1,1_Marimuthu_2025_tmpC.csv,SMILES,tmp/ºC
1,2,2_Huang_2025_mp.csv,SMILES,Melt_C
2,3,3_Tetko_2016_ochem.parquet,SMILES,"Melting Point {measured, converted}"
3,4,4_Austermeier_2025_processed.csv,SMILES,MP


,Source_file,Source,Input_rows,Usable_MP_rows,Excluded_MP_rows,Range_midpoints
0,1_Marimuthu_2025_tmpC.csv,1,7646,7051,595,0
1,2_Huang_2025_mp.csv,2,237406,237406,0,0
2,3_Tetko_2016_ochem.parquet,3,275133,275045,88,59
3,4_Austermeier_2025_processed.csv,4,19609,19609,0,0


Usable observations before structure processing: 539,111


In [6]:
unique_raw_smiles = observations['Original_SMILES'].drop_duplicates().tolist()
structure_results = [canonicalize_and_screen(smiles) for smiles in unique_raw_smiles]
structure_lookup = pd.DataFrame(
    structure_results,
    columns=['SMILES', 'ContainsHalogen', 'ContainsMetal', 'NumFragments', 'MW'],
)
structure_lookup.insert(0, 'Original_SMILES', unique_raw_smiles)
observations = observations.merge(structure_lookup, on='Original_SMILES', how='left', validate='many_to_one')
invalid_structure = observations['SMILES'].isna()
metal_structure = observations['ContainsMetal'].fillna(False)
multicomponent_structure = observations['NumFragments'].fillna(0).ne(1)
broad_organic = ~invalid_structure & ~metal_structure & ~multicomponent_structure
structure_audit = pd.DataFrame({
    'Reason': ['Invalid/non-round-trippable SMILES', 'Contains metal', 'Multiple components',
               'Broad-organic observations retained'],
    'Count': [int(invalid_structure.sum()), int(metal_structure.sum()),
              int(multicomponent_structure.sum()), int(broad_organic.sum())],
})
display(structure_audit)
observations = observations.loc[broad_organic].copy()
observations['Source'] = observations['Source'].astype('int8')
rows_before_exact_deduplication = len(observations)
observations = observations.drop_duplicates(
    ['SMILES', 'MP_reported', 'Source', 'Source_file']
).reset_index(drop=True)
print(f'Exact observation duplicates removed: '
      f'{rows_before_exact_deduplication - len(observations):,}')
print(f'Unique broad-organic canonical SMILES: {observations["SMILES"].nunique():,}')

,Reason,Count
0,Invalid/non-round-trippable SMILES,3
1,Contains metal,105
2,Multiple components,928
3,Broad-organic observations retained,538145


Exact observation duplicates removed: 0
Unique broad-organic canonical SMILES: 307,726


In [7]:
def select_dominant_cluster(group, max_span=5.0, minimum_fraction=2/3):
    ordered = group.sort_values('MP_reported', kind='stable')
    best_index, best_score = None, None
    for start in range(len(ordered)):
        for stop in range(start + 1, len(ordered) + 1):
            candidate = ordered.iloc[start:stop]
            spread = candidate['MP_reported'].max() - candidate['MP_reported'].min()
            if spread > max_span:
                break
            score = (candidate['Source'].nunique(), len(candidate), -spread)
            if best_score is None or score > best_score:
                best_score, best_index = score, candidate.index
    if best_index is None:
        return None
    candidate = group.loc[best_index]
    supported = (len(candidate) >= 2 and candidate['Source'].nunique() >= 2
                 and len(candidate) / len(group) >= minimum_fraction)
    return candidate.index if supported else None

consensus_rows, conflict_audit_rows = [], []
for smiles, group in observations.groupby('SMILES', sort=False):
    group = group.copy()
    all_values = group['MP_reported'].to_numpy(dtype=float)
    all_spread = float(all_values.max() - all_values.min())
    has_multiple_mp = group['MP_reported'].nunique() > 1
    if all_spread <= 10:
        accepted_index = group.index
        method = 'single_measurement' if len(group) == 1 else 'median_all'
    else:
        accepted_index = select_dominant_cluster(group)
        method = 'dominant_cluster' if accepted_index is not None else 'unresolved'
    if accepted_index is not None:
        accepted = group.loc[accepted_index]
        values = accepted['MP_reported'].to_numpy(dtype=float)
        sources = sorted({int(source) for source in accepted['Source']})
        quality = quality_from_values(values, len(sources))
        median = float(np.median(values))
        consensus_rows.append({
            'SMILES': smiles, 'MP': median, 'Source': sources,
            'MP_mean': float(np.mean(values)),
            'MP_std': float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
            'MP_mad': float(np.median(np.abs(values - median))),
            'MP_min': float(np.min(values)), 'MP_max': float(np.max(values)),
            'MP_range': float(np.max(values) - np.min(values)),
            'N_MP_values': len(values), 'N_distinct_MP': int(accepted['MP_reported'].nunique()),
            'N_sources': len(sources), 'MP_quality': quality, 'Consensus_method': method,
            'Has_excluded_values': len(accepted) < len(group),
            'N_excluded_values': len(group) - len(accepted), 'All_MP_range': all_spread,
            'ContainsHalogen': bool(group['ContainsHalogen'].iloc[0]),
            'MW': float(group['MW'].iloc[0]),
        })
    else:
        quality = 'unresolved'
    if has_multiple_mp:
        accepted_set = set(accepted_index) if accepted_index is not None else set()
        for idx, observation in group.iterrows():
            conflict_audit_rows.append({
                'SMILES': smiles, 'MP_reported': float(observation['MP_reported']),
                'MP_raw': observation['MP_raw'], 'Source': int(observation['Source']),
                'Source_file': observation['Source_file'],
                'Accepted_for_consensus': idx in accepted_set, 'MP_quality': quality,
                'Consensus_method': method, 'Group_MP_range': all_spread,
            })
consensus = pd.DataFrame(consensus_rows)
conflict_audit = pd.DataFrame(conflict_audit_rows)
unresolved_count = conflict_audit.loc[
    conflict_audit['MP_quality'].eq('unresolved'), 'SMILES'
].nunique()
print(f'Resolved canonical compounds before final MP/MW filters: {len(consensus):,}')
print(f'Unresolved canonical compounds: {unresolved_count:,}')

Resolved canonical compounds before final MP/MW filters: 307,386
Unresolved canonical compounds: 340


In [8]:
in_mp_range = consensus['MP'].between(0, 500, inclusive='both')
below_mw_limit = consensus['MW'].lt(MAX_MW)
removed_by_mp = int((~in_mp_range).sum())
removed_by_mw_after_mp = int((in_mp_range & ~below_mw_limit).sum())

combined_data = (
    consensus.loc[in_mp_range & below_mw_limit]
    .sort_values(['SMILES', 'MP'], kind='stable').reset_index(drop=True)
)
conflict_audit = conflict_audit.sort_values(
    ['SMILES', 'MP_reported', 'Source'], kind='stable'
).reset_index(drop=True)

def write_combined_parquet(frame, path):
    table = pa.Table.from_pandas(frame, preserve_index=False)
    source_index = table.column_names.index('Source')
    table = table.set_column(
        source_index, 'Source',
        pa.array(frame['Source'].tolist(), type=pa.list_(pa.int8()))
    )
    pq.write_table(table, path, compression='zstd')

write_combined_parquet(combined_data, OUTPUT_DATA_BEFORE_STEREO_CLEANUP)
conflict_audit.to_csv(OUTPUT_CONFLICTS, index=False)

quality_summary = (
    combined_data['MP_quality'].value_counts()
    .reindex(['single', 'exact', 'near_exact', 'high_confidence', 'review'], fill_value=0)
    .rename_axis('MP_quality').reset_index(name='Compounds')
)
display(quality_summary)
print(f'Resolved compounds before final filters: {len(consensus):,}')
print(f'Removed outside 0–500 °C: {removed_by_mp:,}')
print(f'Removed with MW >= {MAX_MW:,.0f} Da after MP filtering: {removed_by_mw_after_mp:,}')
print(f'Compounds before the stereo clean-up: {len(combined_data):,}')
print(f'Halogen-containing compounds retained: {int(combined_data["ContainsHalogen"].sum()):,}')
print(f'Final MW range: {combined_data["MW"].min():.3f}–{combined_data["MW"].max():.3f} Da')
print(f'Wrote: {OUTPUT_DATA_BEFORE_STEREO_CLEANUP.relative_to(DATA_DIR.parent)}')
print(f'Wrote: {OUTPUT_CONFLICTS.relative_to(DATA_DIR.parent)}')


,MP_quality,Compounds
0,single,94754
1,exact,203921
2,near_exact,3209
3,high_confidence,2221
4,review,403


Resolved compounds before final filters: 307,386
Removed outside 0–500 °C: 2,704
Removed with MW >= 1,000 Da after MP filtering: 174
Compounds before the stereo clean-up: 304,508
Halogen-containing compounds retained: 123,975
Final MW range: 18.015–997.694 Da
Wrote: 0_data/outputs/combined_data_before_stereo_cleanup.parquet
Wrote: 0_data/outputs/multiple_MP_compounds.csv


## Validate the saved outputs

The chemistry checks re-parse every saved SMILES: canonical round trip, single fragment, no metals, and the stored halogen flag and molecular weight.

In [10]:
saved_table = pq.read_table(OUTPUT_DATA_BEFORE_STEREO_CLEANUP)
saved = saved_table.to_pandas()
saved_conflicts = pd.read_csv(OUTPUT_CONFLICTS)

expected_columns = [
    'SMILES', 'MP', 'Source', 'MP_mean', 'MP_std', 'MP_mad', 'MP_min', 'MP_max',
    'MP_range', 'N_MP_values', 'N_distinct_MP', 'N_sources', 'MP_quality',
    'Consensus_method', 'Has_excluded_values', 'N_excluded_values',
    'All_MP_range', 'ContainsHalogen', 'MW',
]
assert saved.columns.tolist() == expected_columns
assert saved['SMILES'].is_unique
assert saved['MP'].between(0, 500, inclusive='both').all()
assert saved['MW'].lt(MAX_MW).all()
assert set(saved['MP_quality']).issubset(
    {'single', 'exact', 'near_exact', 'high_confidence', 'review'}
)
assert pa.types.is_list(saved_table.schema.field('Source').type)
assert pa.types.is_int8(saved_table.schema.field('Source').type.value_type)
assert saved['Source'].map(lambda values: list(values) == sorted(set(values))).all()
assert saved['Source'].map(lambda values: set(values).issubset({1, 2, 3, 4})).all()
assert (saved['N_sources'] == saved['Source'].map(len)).all()
assert pd.api.types.is_integer_dtype(saved_conflicts['Source'])

for smiles, contains_halogen, stored_mw in zip(
    saved['SMILES'], saved['ContainsHalogen'], saved['MW']
):
    mol = Chem.MolFromSmiles(smiles)
    assert mol is not None
    assert Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True) == smiles
    assert len(Chem.GetMolFrags(mol)) == 1
    atomic_numbers = {atom.GetAtomicNum() for atom in mol.GetAtoms()}
    assert not (atomic_numbers & METAL_ATOMIC_NUMBERS)
    assert bool(atomic_numbers & HALOGEN_ATOMIC_NUMBERS) == bool(contains_halogen)
    assert np.isclose(Descriptors.MolWt(mol), stored_mw, rtol=0, atol=1e-9)

display(saved.head())
print('Saved schema:')
print(saved_table.schema)
print('All validation checks passed.')

,SMILES,MP,Source,MP_mean,MP_std,MP_mad,MP_min,MP_max,MP_range,N_MP_values,N_distinct_MP,N_sources,MP_quality,Consensus_method,Has_excluded_values,N_excluded_values,All_MP_range,ContainsHalogen,MW
0,B1Oc2ccccc2O1,12.0,[4],12.0,NaN,0.0,12.0,12.0,0.0,1,1,1,single,single_measurement,False,0,0.0,False,119.916
1,BBBBBBBBBB,101.0,[4],101.0,NaN,0.0,101.0,101.0,0.0,1,1,1,single,single_measurement,False,0,0.0,False,120.216
2,B[N+](C)(C)C,95.5,[1],95.5,NaN,0.0,95.5,95.5,0.0,1,1,1,single,single_measurement,False,0,0.0,False,71.940
3,Br/C(=C/c1ccccc1)c1ccc(Br)c2c(Br)cccc12,171.0,[4],171.0,NaN,0.0,171.0,171.0,0.0,1,1,1,single,single_measurement,False,0,0.0,True,466.998
4,Br/C(=N/Nc1ccccc1)c1ccccc1,190.0,[4],190.0,NaN,0.0,190.0,190.0,0.0,1,1,1,single,single_measurement,False,0,0.0,True,275.149


Saved schema:
SMILES: string
MP: double
Source: list<element: int8>
  child 0, element: int8
MP_mean: double
MP_std: double
MP_mad: double
MP_min: double
MP_max: double
MP_range: double
N_MP_values: int64
N_distinct_MP: int64
N_sources: int64
MP_quality: string
Consensus_method: string
Has_excluded_values: bool
N_excluded_values: int64
All_MP_range: double
ContainsHalogen: bool
MW: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 2338
All validation checks passed.
